# 00. 환경 점검 · 데이터 준비 시작

| 부 | 노트북 | DB |
|---|---|---|
| 1부 원리 | 10 NSW · 11 HNSW · 12 IVF · 13 PQ | 안 씀 (numpy 로 직접 구현) |
| 2부 pgvector | 20 데이터·정답지 · 21 HNSW · 22 IVFFlat · 23 양자화 · 24 필터 | 합성 벡터 10만 개 |
| 3부 품질 | 30 퓨전 · 31 분포 진단 | KLUE-MRC (4-1 과 같은 데이터) |

4-1 과 **같은 DB** 를 쓴다(4-1의 docker-compose 하나로 db · pgAdmin · Jupyter 가 함께 뜬다).

In [ ]:
from labutil import *

conn = connect()
q(conn, """
SELECT version()                                              AS postgres,
       (SELECT extversion FROM pg_extension WHERE extname = 'vector')        AS pgvector,
       (SELECT extversion FROM pg_extension WHERE extname = 'textsearch_ko') AS textsearch_ko,
       current_setting('maintenance_work_mem')                AS maintenance_work_mem,
       current_setting('max_parallel_maintenance_workers')    AS 병렬빌드_워커
""").T

`pgvector 0.8.x` 이상이어야 24번(필터 + iterative scan)이 동작한다.

## KLUE 데이터 준비를 백그라운드로 시작
3부에 필요한 것: 텍스트 → BM25 역색인 → 지문 임베딩 → **청크 임베딩(4-2 에서 추가)**
- 4-1 을 마쳤다면 앞의 세 단계는 이미 있어서 건너뛰고, 청크 임베딩만 한다 (약 7분)
- 처음이라면 전부 한다 (약 10분)

코드는 이 폴더의 `prepare_klue.py`. 4-1의 `loader/load.py` 와 `sql/20~22` 를 그대로 재사용한다.

In [ ]:
import subprocess
subprocess.Popen("python prepare_klue.py > prepare.log 2>&1", shell=True)
print("시작했다. 진행 상황은 아래 셀을 다시 실행해서 확인한다.")

In [ ]:
# 진행 상황 확인 (마지막 줄이 '✔ KLUE 준비 완료' 면 끝)
print(open("prepare.log", encoding="utf-8").read()[-1500:])

기다리지 말고 **10번 노트북**으로 넘어간다. 1부는 DB 를 쓰지 않는다.

### 문제가 생기면
- `prepare.log` 에 `Connection refused` → 터미널(4-1 폴더)에서 `docker compose ps` 로 db 가 healthy 인지 확인
- 처음부터 다시: `!python prepare_klue.py --force > prepare.log 2>&1`